# Logistic Regression — sigmoid, cross-entropy, softmax

> Tutorial pair for [`logistic_regression.py`](logistic_regression.py).

## 1. Intuition
Linear regression outputs any real number — useless for "is this spam (0/1)?".
We squash the linear score through a **sigmoid** so the output is a probability
in $(0,1)$, then train it to assign high probability to the correct class. It is
*the* single-neuron classifier and the building block of every neural net.

## 2. Concept (the slide)
- **Model (binary):** $p=\sigma(z),\ z=\mathbf w^\top\mathbf x+b,\ \sigma(z)=\frac{1}{1+e^{-z}}$.
- **Decision:** predict class 1 if $p\ge 0.5$ (i.e. $z\ge 0$) — a *linear* boundary.
- **Loss:** binary cross-entropy (negative log-likelihood of a Bernoulli).
- **Multiclass:** replace sigmoid with **softmax** over $K$ logits; loss is
  categorical cross-entropy.

## 3. Math derivation

**Likelihood.** Each label is Bernoulli: $P(y\mid\mathbf x)=p^{y}(1-p)^{1-y}$.
The negative log-likelihood over the data is

$$\mathcal{L}=-\frac1n\sum_i\big[y_i\log p_i+(1-y_i)\log(1-p_i)\big].$$

**The clean gradient.** Use $\sigma'(z)=\sigma(z)(1-\sigma(z))$. For one example,

$$\frac{\partial \ell}{\partial z}
 =-\Big(\frac{y}{p}-\frac{1-y}{1-p}\Big)\,p(1-p)
 = p-y.$$

So, exactly like linear regression but with $p$ in place of $\hat y$,

$$\boxed{\;\nabla_{\mathbf w}\mathcal{L}=\frac1n X^\top(\mathbf p-\mathbf y),\qquad
 \frac{\partial\mathcal L}{\partial b}=\frac1n\sum_i(p_i-y_i).\;}$$

There is **no closed form** (the equations are transcendental) → we use gradient
descent / Newton's method (IRLS).

**Softmax generalization.** With logits $\mathbf z=W^\top\mathbf x+\mathbf b$ and
$p_k=\dfrac{e^{z_k}}{\sum_j e^{z_j}}$, cross-entropy $-\log p_{y}$ has the same
elegant gradient $\nabla_{\!W}\mathcal L=\frac1nX^\top(P-Y)$ where $Y$ is one-hot.
This "predicted minus target" form is why these losses pair so naturally with
their output nonlinearities.

## 4. NumPy implementation

In [ ]:
# ===== actual implementation from logistic_regression.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _sigmoid(z):
    # numerically stable sigmoid
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def _softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)        # stability
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

import torch

import torch.nn as nn

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    from sklearn.datasets import make_classification, make_blobs

    # --- binary ---
    Xb, yb = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                 n_clusters_per_class=1, random_state=SEED)
    mu, sd = Xb.mean(0), Xb.std(0); Xb = (Xb - mu) / sd
    m = LogisticRegressionNumPy(lr=0.5, n_iters=2000).fit(Xb, yb)
    acc = np.mean(m.predict(Xb) == yb)
    t = LogisticRegressionTorch(2, n_classes=2).fit(Xb, yb, lr=0.5, n_iters=2000)
    acc_t = np.mean(t.predict(Xb) == yb)
    print(f"[binary]  NumPy acc={acc:.3f}   Torch acc={acc_t:.3f}")

    # --- multiclass (softmax) ---
    Xm, ym = make_blobs(n_samples=450, centers=3, n_features=2, random_state=SEED)
    mu, sd = Xm.mean(0), Xm.std(0); Xm = (Xm - mu) / sd
    ms = LogisticRegressionNumPy(multi_class=True, n_classes=3, lr=0.5, n_iters=2000).fit(Xm, ym)
    acc_s = np.mean(ms.predict(Xm) == ym)
    ts = LogisticRegressionTorch(2, n_classes=3).fit(Xm, ym, lr=0.5, n_iters=2000)
    acc_ts = np.mean(ts.predict(Xm) == ym)
    print(f"[softmax] NumPy acc={acc_s:.3f}   Torch acc={acc_ts:.3f}")


class LogisticRegressionNumPy:
    r"""
    Binary:   p = σ(Xw + b),  loss = -Σ[y log p + (1-y) log(1-p)] / n
              gradient:  dL/dw = (1/n) X^T (p - y)   <- same shape as linear reg!
    Softmax:  P = softmax(XW + b),  loss = -Σ log P[i, y_i] / n
              gradient:  dL/dW = (1/n) X^T (P - Y_onehot)
    """

    def __init__(self, multi_class=False, n_classes=None,
                 lam=0.0, lr=0.1, n_iters=2000):
        self.multi_class = multi_class
        self.n_classes = n_classes
        self.lam, self.lr, self.n_iters = lam, lr, n_iters
        self.W = None; self.b = None
        self.history = []

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y)
        n, d = X.shape
        if self.multi_class:
            K = self.n_classes or int(y.max() + 1)
            self.W = np.zeros((d, K)); self.b = np.zeros(K)
            Y = np.eye(K)[y]                                  # one-hot
            for _ in range(self.n_iters):
                P = _softmax(X @ self.W + self.b)
                gW = X.T @ (P - Y) / n + self.lam * self.W
                gb = (P - Y).mean(0)
                self.W -= self.lr * gW; self.b -= self.lr * gb
                self.history.append(-np.mean(np.log(P[np.arange(n), y] + 1e-12)))
        else:
            self.W = np.zeros(d); self.b = 0.0
            y = y.astype(float)
            for _ in range(self.n_iters):
                p = _sigmoid(X @ self.W + self.b)
                gW = X.T @ (p - y) / n + self.lam * self.W
                gb = (p - y).mean()
                self.W -= self.lr * gW; self.b -= self.lr * gb
                eps = 1e-12
                self.history.append(-np.mean(y*np.log(p+eps) + (1-y)*np.log(1-p+eps)))
        return self

    def predict_proba(self, X):
        X = np.asarray(X, float)
        if self.multi_class:
            return _softmax(X @ self.W + self.b)
        return _sigmoid(X @ self.W + self.b)

    def predict(self, X):
        P = self.predict_proba(X)
        return P.argmax(1) if self.multi_class else (P >= 0.5).astype(int)

## 5. PyTorch implementation
`BCEWithLogitsLoss` / `CrossEntropyLoss` fuse the sigmoid/softmax with the log for numerical stability.

In [ ]:
# ===== actual implementation from logistic_regression.py =====
class LogisticRegressionTorch(nn.Module):
    def __init__(self, in_features, n_classes=2):
        super().__init__()
        self.multi = n_classes > 2
        out = n_classes if self.multi else 1
        self.linear = nn.Linear(in_features, out)

    def forward(self, x):
        z = self.linear(x)
        return z if self.multi else z.squeeze(-1)

    def fit(self, X, y, lr=0.1, n_iters=2000):
        dev = get_device(); self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        if self.multi:
            y = torch.as_tensor(y, dtype=torch.long, device=dev)
            loss_fn = nn.CrossEntropyLoss()
        else:
            y = torch.as_tensor(y, dtype=torch.float32, device=dev)
            loss_fn = nn.BCEWithLogitsLoss()
        opt = torch.optim.SGD(self.parameters(), lr=lr)
        for _ in range(n_iters):
            opt.zero_grad()
            loss = loss_fn(self(X), y)
            loss.backward(); opt.step()
        return self

    @torch.no_grad()
    def predict(self, X):
        dev = next(self.parameters()).device
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        z = self(X)
        if self.multi:
            return z.argmax(1).cpu().numpy()
        return (torch.sigmoid(z) >= 0.5).long().cpu().numpy()

## 6. Train — binary and softmax, NumPy vs PyTorch

In [ ]:
demo()

## 7. Visualization — decision boundary & loss curve

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_classification
import logistic_regression as M

X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, random_state=0)
X = (X - X.mean(0)) / X.std(0)
m = M.LogisticRegressionNumPy(lr=0.5, n_iters=2000).fit(X, y)

xx, yy = np.meshgrid(np.linspace(*[X[:,0].min()-1, X[:,0].max()+1], 200),
                     np.linspace(*[X[:,1].min()-1, X[:,1].max()+1], 200))
grid = np.c_[xx.ravel(), yy.ravel()]
proba = m.predict_proba(grid).reshape(xx.shape)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].contourf(xx, yy, proba, levels=20, cmap="RdBu", alpha=.7)
ax[0].scatter(X[:,0], X[:,1], c=y, edgecolor="k", s=15, cmap="RdBu")
ax[0].set_title("P(y=1) and the linear boundary")
ax[1].plot(m.history); ax[1].set_xlabel("iter"); ax[1].set_ylabel("cross-entropy")
ax[1].set_title("Training loss")
plt.tight_layout(); plt.show()

## 8. Takeaways
- Sigmoid+BCE and softmax+CE both give the gradient $\frac1nX^\top(\hat y-y)$ —
  remember this shape.
- The boundary is **linear**; for nonlinear data, add features or hidden layers.
- Always use the *logits* loss (`BCEWithLogitsLoss`) for stability.

**Next:** stack these neurons with nonlinearities → [the MLP](../../dl/mlp/mlp.ipynb).